<a href="https://colab.research.google.com/github/vavan137/Yandex-Cup-2024/blob/main/B.%20From%20Basra%20to%20Damascus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import pandas as pd

In [2]:
with open('/content/map.txt') as f:
  data = json.load(f)

# Нормализуем данные в плоский DataFrame
map = data["Карта"]
prob = data["Вероятности нападения"]
rel = data["Связи между засадами"]

Посмотрю на цепочки уникальных маршрутов

In [3]:
tracks = [['Басра']]
op_cnt=1
while op_cnt>0:
  op_cnt=0
  tracks2=[]
  for cur_tr in tracks:
    cur_ct = cur_tr[-1]
    tmp_l = []
    for ch in map:
      if ch[0]==cur_ct:
        tmp_l.append(ch[1])
        op_cnt+=1
    for y in tmp_l:
      tracks2.append(cur_tr+[y])
  if op_cnt>0:
    tracks = tracks2
#
print(len(tracks))

9


Просто посмотрю, есть ли вероятность путешествия без нападений

In [4]:
max_p0 = -1
opt_track = -1
for tr in tracks:
  p0=1
  for ct in tr:
    p0*=(1-prob.get(ct,0))
  if p0>max_p0:
    max_p0 = p0
    opt_track = tr
  #max_p0 = max(max_p0,p0)
print(max_p0)
print(opt_track)

0.009689043419999998
['Басра', 'Аль-Кадисия', 'Джабаль-Нур', 'Аль-Фаллуджа', 'Джебель-Друз', 'Аль-Мафрак', 'Джабаль-Умм', 'Аль-Хамадия', 'Дамаск']


С учетом того, что караван может выдержать до 3 нападений:
Рассмотрим все цепочки из p и q = (1-p) длины маршрута.
Благоприятные исходы - цепочки, где количество p не более 3

In [9]:
def generate_pq_seq(length,current = ''):
    if length==0:
        yield current
    else:
        yield from generate_pq_seq(length-1,current+'p')
        yield from generate_pq_seq(length-1,current+'q')

def pq_seq_prob(s,tr,data_prob,data_rel):
  prob = 1
  p_cnt = 0
  for i in range(len(s)):
    p_add=0
    for j in range(len(s)):
      s_key = "('" + tr[j] + "', '" + tr[i] + "')"
      if s[j]=='p' and s_key in data_rel:
        p_add+=data_rel[s_key]

    if s[i]=='p':
      p_cnt+=1
      prob*= data_prob.get(tr[i],0)+p_add
    elif s[i]=='q':
      prob*=(1-data_prob.get(tr[i],0)-p_add)
  return prob,p_cnt

In [10]:
max_prob = -1
opt_track = -1

for tr in tracks:
  sum_pr = 0
  good_pr = 0
  seq = list(generate_pq_seq(len(tr)))
  for s in seq:
    a,b = pq_seq_prob(s,tr,prob,rel)
    if b<=3:
      sum_pr+=a
      good_pr+=a
    else:
      sum_pr+=a
  if good_pr/sum_pr>max_prob:
    max_prob = good_pr/sum_pr
    opt_track = tr

print(max_prob)
print(opt_track)

0.4197084858584149
['Басра', 'Аль-Кадисия', 'Джабаль-Нур', 'Аль-Фаллуджа', 'Аль-Касра', 'Тель-Калах', 'Вади-Муса', 'Аль-Кусейма', 'Дамаск']


Максимальная вероятность успешного путешествия 0.42